# IDX Cold Archive Migration (maintenance)
Bridge: staging → Google Drive. Not a trading runtime.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive'
except Exception as e:
    DRIVE_ROOT = '/tmp/idx_drive_sim'
    print('sim root', DRIVE_ROOT, e)


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from src.python.archive.drive_fs_backend import DriveFsBackend
from src.python.archive.migration import MigrationEngine, discover_local_sources
drive = DriveFsBackend(Path(DRIVE_ROOT))
eng = MigrationEngine(drive=drive, journal_path=Path(DRIVE_ROOT)/'IDX/cold_archive/migration_journal.json')
print('available', drive.available())


In [ ]:
SOURCE_DIR = Path('staging_archives')
sources = discover_local_sources(SOURCE_DIR)
for s in sources:
    data = Path(s['path']).read_bytes()
    rec = eng.migrate_bytes(data, archive_id=s['archive_id'], filename=s['filename'], source='github')
    print(rec.archive_id, rec.status, rec.error_code)
print(eng.inventory(sources))
print(eng.fifo_rotate())
